# pandas：欠損値・検出器別集計・保存

対応する本文は [pandas による表形式データ解析](../chapters/pandas.tex) である。コードセルを上から順にShift+Enterで実行する。各問の説明にある値や図と、セルの下に表示される結果を比べる。ファイルを作る問では、その直前に保存先を用意する。

## 入力CSVを用意する

本文の6件の記録を `events.csv` に保存し、pandasで読む。`signal` が空欄の行は1件ある。まず保存先を作り、次のセルに表示される表が6行4列になっていることを確認する。

### 保存先を用意する

この後のセルではファイルを作る。次のセルを実行して、`notebooks/_outputs/06_pandas/` を用意する。表示された場所に結果が保存され、同名の実習用ファイルは再実行すると置き換わる。

Notebookは教材の `notebooks` フォルダー、または教材の一番上のフォルダーから実行する。`Path.cwd()` は現在のフォルダーを調べ、`mkdir()` は保存先を作る。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "main.tex").is_file():
    raise FileNotFoundError("教材のフォルダーか notebooks フォルダーから実行してください。")
WORK = ROOT / "notebooks" / "_outputs" / "06_pandas"
WORK.mkdir(parents=True, exist_ok=True)
print("Python", sys.version.split()[0])
print("保存先:", WORK.relative_to(ROOT))

Python 3.12.14
保存先: notebooks/_outputs/06_pandas


In [2]:
import numpy as np
import pandas as pd

CSV_PATH = WORK / "events.csv"
CSV_PATH.write_text('detector,time_ns,signal,valid\nA,1000,12.4,true\nB,1180,9.8,true\nA,1300,15.1,true\nB,1510,,false\nA,1760,11.0,true\nB,2050,13.2,true\n', encoding="utf-8")
df = pd.read_csv(CSV_PATH)
df

,detector,time_ns,signal,valid
0,A,1000,12.4,True
1,B,1180,9.8,True
2,A,1300,15.1,True
3,B,1510,NaN,False
4,A,1760,11.0,True
5,B,2050,13.2,True


## 問1：表の構造を確認する

`head(3)` は先頭3行、`isna().sum()` は行方向を集計し、列ごとの欠損数を返す。空欄のある信号列は浮動小数点、`valid` は真偽値として読む。型推論の結果は必ず確認する。

列の値から種類を自動で選ぶことを型推論という。ここでは `signal` の欠損が `NaN` と表示される。

この後のセルにある `assert 条件` は、結果を自動で確認する命令である。条件が違っていればエラーで止まり、条件を満たせば次へ進む。何も表示されなくても、エラーが出なければその確認は通っている。画面の数値や図も、説明に示した値と比べる。

In [3]:
print(df.shape, df.columns.tolist())
print(df.dtypes)
print(df.head(3))
print(df.isna().sum())
assert df.shape == (6, 4) and df.signal.isna().sum() == 1
assert pd.api.types.is_bool_dtype(df.valid)

(6, 4) ['detector', 'time_ns', 'signal', 'valid']
detector        str
time_ns       int64
signal      float64
valid          bool
dtype: object
  detector  time_ns  signal  valid
0        A     1000    12.4   True
1        B     1180     9.8   True
2        A     1300    15.1   True
detector    0
time_ns     0
signal      1
valid       0
dtype: int64


## 問2：信号値が12以上の有効な行を選ぶ

まず各行が条件を満たすかを表示する。`df["valid"]` は有効かどうかの列、`df["signal"] >= 12.0` は信号値の比較である。`&` で両方を満たす行を選ぶ。`and` では表全体を一つの真偽値として扱おうとするので、ここでは使わない。

In [4]:
mask = df["valid"] & (df["signal"] >= 12.0)
print(mask)

0     True
1    False
2     True
3    False
4    False
5     True
dtype: bool


0、2、5行目がTrueとなる。`loc` へこの条件を渡すと3行が残る。次は、その3行の信号値を抽出後の平均で割る。`copy()` は元の表から独立した部分表を作る。

In [5]:
selected = df.loc[mask].copy()
selected_mean = selected["signal"].mean()
print("抽出後の平均:", selected_mean)
selected["signal_ratio"] = selected["signal"] / selected_mean
print(selected)
assert len(selected) == 3
assert np.isclose(selected.signal_ratio.mean(), 1.0)
assert "signal_ratio" not in df.columns

抽出後の平均: 13.566666666666668
  detector  time_ns  signal  valid  signal_ratio
0        A     1000    12.4   True      0.914005
2        A     1300    15.1   True      1.113022
5        B     2050    13.2   True      0.972973


## 問3：有効信号を集計し、較正係数を結合する

第2問の強信号だけの表は使わず、元の6行から集計する。`count` は信号の有効値の件数を数える。標準偏差は `ddof=1` に相当する。`on="detector"` は検出器名を照合に使う指定、`how="left"` は元の記録を全行残す指定である。例えばAの信号12.4にはAの較正係数1.02が付き、較正後は `12.4 * 1.02 = 12.648` となる。較正表のキーが重複すると行が増えるため、`validate="many_to_one"` で検出する。

In [6]:
valid = df.loc[df.valid & df.signal.notna()]
summary = valid.groupby("detector", as_index=False).agg(
    count=("signal", "count"), mean=("signal", "mean"), std=("signal", "std"))
np.testing.assert_allclose(summary["mean"], [38.5 / 3, 11.5])
np.testing.assert_array_equal(summary["count"], [3, 2])
gain = pd.DataFrame({"detector": ["A", "B"], "gain": [1.02, 0.97]})
calibrated = df.merge(gain, on="detector", how="left", validate="many_to_one")
calibrated["signal_calibrated"] = calibrated.signal * calibrated.gain
assert len(calibrated) == len(df)
print(summary)

  detector  count       mean       std
0        A      3  12.833333  2.084067
1        B      2  11.500000  2.404163


## 問4：チャンクをまたいで合計と件数を足す

`chunksize=2` はCSVを2行ずつ読む指定であり、分けて読んだ一まとまりをチャンクという。各チャンクの平均をそのまま平均すると、件数の異なるチャンクに同じ重みを付けてしまう。各検出器の合計と有効件数を保存し、最後に割る。ここでは第3問と同じ有効条件を使う。

`defaultdict(float)` は、初めて出る検出器の合計を0.0から始める辞書である。`defaultdict(int)` は件数を0から始める。`pd.testing.assert_frame_equal()` は、まとめて読んだ表と分割して読んだ表の値・列・型を比べる。

In [7]:
from collections import defaultdict

totals, numbers = defaultdict(float), defaultdict(int)
for chunk in pd.read_csv(CSV_PATH, chunksize=2):
    chunk = chunk.loc[chunk.valid & chunk.signal.notna()]
    for detector, group in chunk.groupby("detector"):
        totals[detector] += group.signal.sum()
        numbers[detector] += group.signal.count()
chunk_summary = pd.DataFrame([
    {"detector": key, "count": numbers[key], "mean": totals[key] / numbers[key]}
    for key in sorted(numbers)])
pd.testing.assert_frame_equal(chunk_summary, summary[["detector", "count", "mean"]])
wrong = np.mean([np.mean([0.0]), np.mean([10.0, 20.0, 30.0])])
correct = np.mean([0.0, 10.0, 20.0, 30.0])
assert wrong == 10 and correct == 15
print(chunk_summary, "\nチャンク平均の平均:", wrong, "全件の平均:", correct)

  detector  count       mean
0        A      3  12.833333
1        B      2  11.500000 
チャンク平均の平均: 10.0 全件の平均: 15.0


## 問5：CSV と Parquet を読み戻す

`index=False` は行ラベルの保存を省く。CSV は型を推論して読み、Parquet は型情報を保持する。値に加えて列名、順序、欠損を比較する。Parquet には `pyarrow` を使用する。

Parquetは、表の値と列の型を保存するバイナリ形式である。pyarrowはこの形式の読み書きに使うライブラリである。

In [8]:
summary.to_csv(WORK / "summary.csv", index=False)
summary.to_parquet(WORK / "summary.parquet", index=False, engine="pyarrow")
for path in [WORK / "summary.csv", WORK / "summary.parquet"]:
    loaded = pd.read_csv(path) if path.suffix == ".csv" else pd.read_parquet(path)
    pd.testing.assert_frame_equal(loaded, summary, check_exact=False, rtol=1e-12, atol=1e-12)
    print(path.name, "を照合しました。")

summary.csv を照合しました。
summary.parquet を照合しました。
